# Module 4 — NER with XLM-RoBERTa (Google Colab)

**Goal:** find the **places (LOC)** and **dates (DATE)** in a message, by fine-tuning the pretrained model **XLM-RoBERTa base** for *token classification*.
```
"nheb billet men sidi bou said l Paris ghodwa"
                 ─────────────   ─────  ──────
                      LOC         LOC    DATE
```

### How to run this notebook
1. In Colab: **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. **Runtime → Run all**. Everything runs by itself (~10–15 minutes): installs the libraries, downloads our data from GitHub, trains, evaluates.
3. At the end, a file **`ner_xlmr.zip`** is downloaded (the trained model + its metrics). Unzip it into the project folder `models/ner/` on your PC.

Nothing needs to be uploaded: the data is read directly from the public GitHub repo.

## 0. Setup

- Check that a **GPU** is available (training on CPU would take hours).
- Install the exact library versions used in the project, so the saved model loads on the local PC without surprises.
- Fix the random seeds (42) so the run is reproducible.

In [ ]:
!pip install -q "transformers==5.24.0" "seqeval==1.2.2" accelerate

import json, random, re, urllib.request
import numpy as np
import torch
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments, set_seed)

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
set_seed(42)

## 1. Load the data from GitHub

The same files as every other module:
- **train** (3,161 messages, travel + non-travel): the model learns from them. Non-travel messages teach it "no entity here" too.
- **val** (684 messages): to measure the model after each epoch and keep the best one.
- **challenge set** (60 messages): places and dates that are **not in our lexicons** (Kerkennah, La Marsa, Nice, "dans trois jours", "el chta hedhi"...). Train, val and test only contain the 43 cities and date phrases of our lexicons, so they overestimate how well the model handles the real world; the challenge set is the honest test.
- The **test** set is not used here (final evaluation only).

We use the **original `text`** (not masked, not lowercased): XLM-R is case-sensitive (`Paris` is one known token, `paris` is cut into `▁par` + `is`).

In [ ]:
BASE = "https://raw.githubusercontent.com/MohamedBenMaaouia/multilingual-travel-chatbot-/main/"

def load_jsonl(path):
    with urllib.request.urlopen(BASE + path) as f:
        return [json.loads(line) for line in f.read().decode("utf-8").splitlines() if line.strip()]

train_rows = load_jsonl("data/processed/train.jsonl")
val_rows = load_jsonl("data/processed/val.jsonl")
challenge_rows = load_jsonl("data/challenge/ner_challenge.jsonl")
for name, rows in [("train", train_rows), ("val", val_rows), ("challenge", challenge_rows)]:
    n_loc = sum(e["label"] == "LOC" for r in rows for e in r["entities"])
    n_date = sum(e["label"] == "DATE" for r in rows for e in r["entities"])
    print(f"{name:9s} {len(rows):5d} messages | LOC {n_loc:5d} | DATE {n_date:4d}")

## 2. From character spans to one label per word (BIO)

Our data stores entities as **character positions** (`start`, `end`). The model needs **one label per word**, with the **BIO** scheme:
- **B-LOC** = first word of a place, **I-LOC** = following words of the same place;
- **B-DATE**, **I-DATE** = the same for dates;
- **O** = not part of an entity.

```
words:  nheb  billet  men  sidi   bou    said   l  Paris  ghodwa
labels:  O      O      O   B-LOC  I-LOC  I-LOC  O  B-LOC  B-DATE
```

**Splitting into words:** we cut the text into words *and* punctuation marks separately, including apostrophes and hyphens (so `Djerba?` → `Djerba` + `?`, `Tunis-Malte` → `Tunis` + `-` + `Malte`, `d'Istanbul` → `d` + `'` + `Istanbul`), and keep each word's character positions. A word gets an entity label if it lies inside an entity's span. The check below verifies that every entity of the data lines up exactly with word boundaries.

In [ ]:
LABELS = ["O", "B-LOC", "I-LOC", "B-DATE", "I-DATE"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
WORD_RE = re.compile(r"\w+|[^\w\s]")      # words, and every punctuation mark on its own (incl. ' and -)

def split_words(text):
    return [(m.group(), m.start(), m.end()) for m in WORD_RE.finditer(text)]

def bio_labels(text, entities):
    words = split_words(text)
    labels = ["O"] * len(words)
    misaligned = 0
    for e in entities:
        inside = [i for i, (_, s, t) in enumerate(words) if s >= e["start"] and t <= e["end"]]
        covered = "".join(text[words[i][1]:words[i][2]] for i in inside)
        if not inside or covered != re.sub(r"\s+", "", text[e["start"]:e["end"]]):
            misaligned += 1
            continue
        for k, i in enumerate(inside):
            labels[i] = ("B-" if k == 0 else "I-") + e["label"]
    return [w for w, _, _ in words], labels, misaligned

for name, rows in [("train", train_rows), ("val", val_rows), ("challenge", challenge_rows)]:
    bad = sum(bio_labels(r["text"], r["entities"])[2] for r in rows)
    print(f"{name:9s} entities not aligned with word boundaries: {bad}")

words, labels, _ = bio_labels("nheb billet men sidi bou said l Paris ghodwa",
                              [{"start": 16, "end": 29, "label": "LOC"}, {"start": 32, "end": 37, "label": "LOC"},
                               {"start": 38, "end": 44, "label": "DATE"}])
print("\nexample:", list(zip(words, labels)))

## 3. From words to XLM-R's subword tokens (label alignment)

XLM-R cuts words into **subword tokens** from its 250,000-piece vocabulary, so one word can become several tokens (`ghodwa` → `▁gho` + `dwa`). But our labels are per **word**. Standard solution:
- the **first token** of each word gets the word's label;
- the other tokens of the word get **-100**, which means "ignore this token when computing the loss";
- the special tokens `<s>` and `</s>` also get -100.

The table below prints this for one message, so you can see exactly what the model is trained on.

In [ ]:
MODEL_NAME = "FacebookAI/xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(row):
    words, labels, _ = bio_labels(row["text"], row["entities"])
    enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=96)
    aligned, previous = [], None
    for wid in enc.word_ids():
        if wid is None or wid == previous:
            aligned.append(-100)                 # special token or 2nd/3rd piece of a word
        else:
            aligned.append(label2id[labels[wid]])
        previous = wid
    enc["labels"] = aligned
    return enc

class ListDataset(torch.utils.data.Dataset):
    def __init__(self, rows): self.items = [encode(r) for r in rows]
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

train_ds, val_ds, challenge_ds = ListDataset(train_rows), ListDataset(val_rows), ListDataset(challenge_rows)

example = encode({"text": "nheb billet men sidi bou said l Paris ghodwa",
                  "entities": [{"start": 16, "end": 29, "label": "LOC"}, {"start": 32, "end": 37, "label": "LOC"},
                               {"start": 38, "end": 44, "label": "DATE"}]})
print(f"{'token':10s} label")
for tok, lab in zip(tokenizer.convert_ids_to_tokens(example["input_ids"]), example["labels"]):
    print(f"{tok:10s} {'ignored (-100)' if lab == -100 else id2label[lab]}")

## 4. The model

`AutoModelForTokenClassification` = the pretrained XLM-R body (12 Transformer layers, ~270M weights, already trained by Meta on 100 languages) **+ a new head**: one linear layer that turns each token's 768-number vector into **5 scores** (O, B-LOC, I-LOC, B-DATE, I-DATE), then softmax. The head starts random; the body starts with its pretrained knowledge.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS),
                                                        id2label=id2label, label2id=label2id)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f} million weights")

## 5. Training (fine-tuning)

The training loop is the same as for the MLP (forward pass → cross-entropy loss → backpropagation → update), but through all 12 layers, and with settings made for **gentle** adjustments of a pretrained model:

| setting | value | why |
|---|---|---|
| learning rate | 3e-5 | ~30× smaller than the MLP: big steps would erase the pretrained knowledge (*catastrophic forgetting*) |
| warmup | first 10% of steps | start even more gently while the new head is still random |
| epochs | 5 | the model learns fast; more would overfit |
| batch size | 16 | |
| weight decay | 0.01 | AdamW's L2-like penalty against large weights |
| fp16 | yes | 16-bit numbers on the GPU: ~2× faster, half the memory |

After **each epoch** we measure on **val**: loss, and entity-level **precision / recall / F1** (seqeval). At the end, the model from the epoch with the **best val F1** is kept.

**How the metrics are computed (seqeval):** an entity counts as correct only if **both its boundaries and its label** are exactly right (`l Paris` instead of `Paris` = wrong).

In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def to_tag_lists(predictions, label_ids):
    pred_ids = np.argmax(predictions, axis=-1)
    true_tags, pred_tags = [], []
    for p_row, l_row in zip(pred_ids, label_ids):
        t, p = [], []
        for p_id, l_id in zip(p_row, l_row):
            if l_id != -100:                      # only the first token of each word counts
                t.append(id2label[int(l_id)]); p.append(id2label[int(p_id)])
        true_tags.append(t); pred_tags.append(p)
    return true_tags, pred_tags

def compute_metrics(eval_pred):
    true_tags, pred_tags = to_tag_lists(eval_pred.predictions, eval_pred.label_ids)
    return {"precision": precision_score(true_tags, pred_tags), "recall": recall_score(true_tags, pred_tags),
            "f1": f1_score(true_tags, pred_tags)}

args = TrainingArguments(
    output_dir="ner_runs", learning_rate=3e-5, warmup_ratio=0.1, num_train_epochs=5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64, weight_decay=0.01, fp16=True,
    eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
    save_total_limit=1, seed=42, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorForTokenClassification(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## 6. Learning curves and diagnosis (under- / overfitting)

- **Curves:** train loss and val loss per epoch, and val F1 per epoch.
- **The direct check (no curves needed):** F1 on **train** vs F1 on **val** for the final model. Both high and close → good fit; train much higher → overfitting; both low → underfitting.

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
train_loss = [h["loss"] for h in hist if "loss" in h and "eval_loss" not in h]
val_loss = [h["eval_loss"] for h in hist if "eval_loss" in h]
val_f1 = [h["eval_f1"] for h in hist if "eval_f1" in h]
epochs = range(1, len(val_loss) + 1)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(epochs, train_loss[:len(val_loss)], "o-", label="train loss")
a1.plot(epochs, val_loss, "o-", label="val loss")
a1.set(title="Loss per epoch", xlabel="epoch"); a1.legend(); a1.grid(alpha=0.3)
a2.plot(epochs, val_f1, "o-", color="green", label="val entity F1")
a2.set(title="Val entity F1 per epoch", xlabel="epoch", ylim=(min(val_f1) - 0.02, 1.0)); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("ner_learning_curves.png", dpi=150); plt.show()

train_metrics = trainer.evaluate(train_ds, metric_key_prefix="train")
val_metrics = trainer.evaluate(val_ds, metric_key_prefix="val")
print(f"TRAIN entity F1 {train_metrics['train_f1']:.2%} | VAL entity F1 {val_metrics['val_f1']:.2%}")

## 7. Detailed evaluation: val and challenge set

For each set: precision / recall / F1 **per label** (LOC, DATE), F1 **per language**, and the messages where the predicted entities differ from the true ones.

In [ ]:
def predict_entities(text):
    # Run the model on one raw message and return [(entity text, label, start, end)].
    words = split_words(text)
    enc = tokenizer([w for w, _, _ in words], is_split_into_words=True, truncation=True, max_length=96,
                    return_tensors="pt").to(model.device)
    with torch.no_grad():
        pred = model(**enc).logits.argmax(-1)[0].tolist()
    tags, previous = ["O"] * len(words), None
    for pos, wid in enumerate(enc.word_ids()):
        if wid is not None and wid != previous:
            tags[wid] = id2label[pred[pos]]
        previous = wid
    spans = []
    for i, tag in enumerate(tags):
        if tag.startswith("B-") or (tag.startswith("I-") and (not spans or spans[-1][1] != tag[2:] or spans[-1][3] != i - 1)):
            spans.append([words[i][1], tag[2:], words[i][2], i])          # start, label, end, last word index
        elif tag.startswith("I-"):
            spans[-1][2], spans[-1][3] = words[i][2], i
    return [(text[s:e], label, s, e) for s, label, e, _ in spans]

def evaluate_set(name, rows, ds):
    out = trainer.predict(ds)
    true_tags, pred_tags = to_tag_lists(out.predictions, out.label_ids)
    print(f"===== {name.upper()} =====")
    print(classification_report(true_tags, pred_tags, digits=3))
    per_lang = {}
    for lang in ["en", "fr", "tn_latin"]:
        idx = [i for i, r in enumerate(rows) if r["lang"] == lang]
        per_lang[lang] = f1_score([true_tags[i] for i in idx], [pred_tags[i] for i in idx])
    print("entity F1 per language:", {k: round(v, 3) for k, v in per_lang.items()})
    mistakes = []
    for r in rows:
        gold = sorted((r["text"][e["start"]:e["end"]], e["label"]) for e in r["entities"])
        pred = sorted((t, l) for t, l, _, _ in predict_entities(r["text"]))
        if gold != pred:
            mistakes.append({"text": r["text"], "true": gold, "predicted": pred})
    print(f"\n{len(mistakes)} messages with at least one mistake (showing up to 25):")
    for m in mistakes[:25]:
        print(f"  {m['text']}\n      true: {m['true']}\n      pred: {m['predicted']}")
    report = classification_report(true_tags, pred_tags, output_dict=True)
    return {"f1": f1_score(true_tags, pred_tags), "precision": precision_score(true_tags, pred_tags),
            "recall": recall_score(true_tags, pred_tags),
            "per_label": {k: {m: float(v) for m, v in report[k].items()} for k in ["LOC", "DATE"] if k in report},
            "f1_per_language": per_lang, "mistakes": mistakes}

val_results = evaluate_set("val", val_rows, val_ds)
challenge_results = evaluate_set("challenge (places/dates outside our lexicons)", challenge_rows, challenge_ds)

## 8. Save and download

- The model is saved in **16-bit** (half the size, ~560 MB instead of ~1.1 GB); on the PC it is loaded back in 32-bit.
- The tokenizer is saved with it, plus `metrics.json` (settings, train/val/challenge scores, epoch history, mistakes) and the learning-curve figure.
- Everything is zipped into **`ner_xlmr.zip`** and downloaded. **Unzip it into `models/ner/`** in the project.

In [ ]:
import shutil, os
from google.colab import files

SAVE_DIR = "ner_xlmr"
trainer.model.half().save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
metrics = {"model": MODEL_NAME, "labels": LABELS,
           "settings": {"learning_rate": 3e-5, "epochs": 5, "batch_size": 16, "warmup_ratio": 0.1,
                        "weight_decay": 0.01, "max_length": 96, "seed": 42},
           "best_epoch_val_f1": max(val_f1), "train_f1": train_metrics["train_f1"],
           "val": val_results, "challenge": challenge_results, "history": hist}
with open(os.path.join(SAVE_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2, default=float)
shutil.copy("ner_learning_curves.png", SAVE_DIR)
shutil.make_archive("ner_xlmr", "zip", SAVE_DIR)
print(f"ner_xlmr.zip: {os.path.getsize('ner_xlmr.zip') / 1e6:.0f} MB")
files.download("ner_xlmr.zip")